# 第17章　学習がうまくいかないとき ― デバッグと実践的な落とし穴**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 切り分けの早見表 ― 症状から原因へたどる

In [ ]:
common = set(train_df.patient_id) & set(val_df.patient_id)assert not common, f"患者リーク: {len(common)}人が学習と検証の両方に混入"

## 勾配とアクティベーションを覗く ― NaNの発生源を突き止める

In [ ]:
import torchdef attach_nan_hooks(model):    def fwd(name):        def hook(_, __, out):            t = out[0] if isinstance(out, tuple) else out            if torch.is_tensor(t) and not torch.isfinite(t).all():                raise RuntimeError(f"[forward] {name} で非有限値が発生")        return hook    for name, m in model.named_modules():        m.register_forward_hook(fwd(name))     # 最初にNaNを出した層名が分かる

In [ ]:
for name, p in model.named_parameters():    if p.grad is not None:        g = p.grad.norm().item()        if g > 1e3 or g < 1e-8:                # 爆発 or 消失            print(f"{name}: grad_norm={g:.2e}")

## ミスラベルを「見つけて直す」 ― 外れ値・誤ラベルの検出

In [ ]:
def find_label_errors(probs, labels):          # probs: out-of-fold の予測確率 (N,C)    C = probs.shape[1]    thr = np.array([probs[labels == j, j].mean() for j in range(C)])  # クラス別閾値    pred = probs.argmax(1)    conf = probs[np.arange(len(labels)), pred]    suspects = (pred != labels) & (conf >= thr[pred])   # 自信をもって食い違う    return np.where(suspects)[0]

In [ ]:
def aum_update(aum_sum, logits, y, batch_idx):  # batch_idx: この症例が何番目か（Datasetが返す）    z_y = logits.gather(1, y[:, None]).squeeze(1)    other = logits.clone(); other.scatter_(1, y[:, None], float("-inf"))    aum_sum[batch_idx] += (z_y - other.max(1).values).detach().cpu()  # 割当クラス - 最強他クラス    return aum_sum# ※ 症例ごとに積み上げるので、Dataset の __getitem__ は (画像, ラベル, 症例インデックス) を返すようにする

## ラベルが間違っている前提で学ぶ ― ノイズ耐性の技術

In [ ]:
def gce_loss(logits, y, q=0.7, eps=1e-6):    p = F.softmax(logits, dim=1)    fy = p.gather(1, y[:, None]).squeeze(1).clamp_min(eps)   # 真クラスの確率    return ((1.0 - fy.pow(q)) / q).mean()      # q→0でCE、q=1でMAEに連続変化

In [ ]:
def small_loss_mask(losses, keep_ratio):       # keep_ratio: 1-推定ノイズ率へ漸減    k = max(1, int(len(losses) * keep_ratio))    idx = losses.argsort()[:k]                 # 損失の小さい方＝正しい可能性が高い    m = torch.zeros_like(losses, dtype=torch.bool); m[idx] = True    return m

In [ ]:
@torch.no_grad()def update_ema(student, teacher, alpha=0.999):    for pt, ps in zip(teacher.parameters(), student.parameters()):        pt.mul_(alpha).add_(ps, alpha=1 - alpha)   # 教師＝生徒の緩やかなコピー    # parameters() に BatchNorm が保持する平均・分散の移動平均（running statistics、buffers）は含まれない。    # 放置すると教師のBN統計は初期値のままで、教師の予測が壊れる。    for bt, bs in zip(teacher.buffers(), student.buffers()):        bt.copy_(bs)